In [2]:
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path

# 1. Identificar a raiz do projeto
PROJECT_ROOT = Path.cwd().parent

# 2. Definir os caminhos dos artefatos
DATA_PATH = PROJECT_ROOT / "data" / "processed"
MODEL_PATH = PROJECT_ROOT / "models"

# 3. Carregar a base de clientes segmentada
rfm = pd.read_csv(DATA_PATH / "rfm_segmented_customers.csv")

# 4. Carregar os modelos treinados
scaler = joblib.load(MODEL_PATH / "scaler.pkl")
kmeans = joblib.load(MODEL_PATH / "kmeans_model.pkl")
mapa_segmentos = joblib.load(MODEL_PATH / "mapa_segmentos.pkl")

print("--- Artefatos carregados com sucesso! ---")
print(f"Total de clientes na base: {len(rfm)}")
print(f"Segmentos disponíveis: {list(mapa_segmentos.values())}")

--- Artefatos carregados com sucesso! ---
Total de clientes na base: 4338
Segmentos disponíveis: ['Clientes Promissores', 'Clientes VIP', 'Clientes Regulares', 'Clientes Inativos']


In [3]:
print("Dimensões da base:", rfm.shape)

display(rfm.head())

print("\nColunas disponíveis:")
print(rfm.columns.tolist())

print("\nDistribuição dos segmentos:")
print(rfm["Segmento"].value_counts())

Dimensões da base: (4338, 6)


,CustomerID,Recency,Frequency,Monetary,Cluster,Segmento
0,12346.0,326,1,77183.60,3,Clientes Promissores
1,12347.0,2,7,4310.00,0,Clientes VIP
2,12348.0,75,4,1797.24,3,Clientes Promissores
3,12349.0,19,1,1757.55,2,Clientes Regulares
4,12350.0,310,1,334.40,1,Clientes Inativos



Colunas disponíveis:
['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Cluster', 'Segmento']

Distribuição dos segmentos:
Segmento
Clientes Inativos       1622
Clientes Promissores    1166
Clientes Regulares       837
Clientes VIP             713
Name: count, dtype: int64


In [4]:
pip install google-genai

Note: you may need to restart the kernel to use updated packages.


In [9]:
!pip install groq python-dotenv

In [10]:
import os
from dotenv import load_dotenv
from pathlib import Path

# Localiza o arquivo .env na pasta notebooks
env_path = Path.cwd() / ".env"

load_dotenv(env_path, override=True)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if GROQ_API_KEY:
    print("Chave da API carregada com sucesso!")
else:
    print("Chave não encontrada. Verifique o arquivo .env.")

Chave da API carregada com sucesso!


In [13]:
import json
import time
import os
from groq import Groq
from dotenv import load_dotenv, find_dotenv

# 1. Carrega a chave de ambiente
load_dotenv(find_dotenv(), override=True)
minha_chave = os.getenv("GROQ_API_KEY")

if not minha_chave:
    raise ValueError("ERRO: A variável GROQ_API_KEY não foi encontrada no arquivo .env.")
else:
    print("--- Chave da Groq carregada com sucesso! Conectando... ---")

# 2. Inicializa o cliente da Groq
client = Groq(api_key=minha_chave)

# 3. Resumo estatístico dos 4 clusters a partir da base RFM
resumo_clusters = rfm.groupby(['Cluster', 'Segmento']).agg({
    'CustomerID': 'count',
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean'
}).reset_index().rename(columns={'CustomerID': 'Quantidade_Clientes'})

todas_as_personas = {}

# 4. Loop de geração de personas via Groq
for index, row in resumo_clusters.iterrows():
    
    nome_segmento = row['Segmento']
    print(f"\nProcessando IA via Groq para: {nome_segmento}...")
    
    dados_cluster = {
        "segmento": nome_segmento,
        "quantidade_clientes": int(row['Quantidade_Clientes']),
        "recencia_media_dias": round(row['Recency'], 2),
        "frequencia_media_compras": round(row['Frequency'], 2),
        "valor_monetario_medio_rs": round(row['Monetary'], 2)
    }
    
    prompt = f"""
    Você é um especialista sênior em CRM e Marketing de E-commerce.
    Analise os dados estatísticos do seguinte segmento de clientes e gere a persona e estratégias.

    Dados do Segmento:
    {json.dumps(dados_cluster, ensure_ascii=False)}

    Retorne a resposta ESTRITAMENTE em formato JSON com as seguintes chaves:
    - "persona_nome": um título marcante para este perfil
    - "descricao_persona": breve resumo comportamental
    - "estrategia_retencao": ação recomendada de marketing
    - "copy_notificacao_push": texto direto para enviar no celular do cliente
    """
    
    max_tentativas = 3
    for tentativa in range(max_tentativas):
        try:
            # Chamada da API Groq usando a sintaxe chat.completions
            chat_completion = client.chat.completions.create(
                messages=[
                    {
                        "role": "system",
                        "content": "Você é um assistente especialista em CRM que responde estritamente em JSON."
                    },
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                model="openai/gpt-oss-120b",
                temperature=0.2,
                response_format={"type": "json_object"}
            )
            
            # Captura a resposta e converte de string JSON para dicionário
            resposta_texto = chat_completion.choices[0].message.content
            todas_as_personas[nome_segmento] = json.loads(resposta_texto)
            
            print(f"✓ Persona gerada com sucesso para {nome_segmento}!")
            break
            
        except Exception as e:
            print(f"  [Aviso] Falha na tentativa {tentativa + 1}/{max_tentativas}: {e}")
            if tentativa < max_tentativas - 1:
                time.sleep(3)
            else:
                raise e

print("\n--- Processamento concluído com sucesso com a Groq! ---")

--- Chave da Groq carregada com sucesso! Conectando... ---

Processando IA via Groq para: Clientes VIP...
✓ Persona gerada com sucesso para Clientes VIP!

Processando IA via Groq para: Clientes Inativos...
✓ Persona gerada com sucesso para Clientes Inativos!

Processando IA via Groq para: Clientes Regulares...
✓ Persona gerada com sucesso para Clientes Regulares!

Processando IA via Groq para: Clientes Promissores...
✓ Persona gerada com sucesso para Clientes Promissores!

--- Processamento concluído com sucesso com a Groq! ---


In [16]:
import json
import joblib


# 1. Salva o dicionário gerado pela IA no disco
with open('personas.json', 'w', encoding='utf-8') as f:
    json.dump(todas_as_personas, f, ensure_ascii=False, indent=4)

print("--- Arquivo personas.json gerado com sucesso no disco! ---")

# 2. Salva o resumo estatístico dos clusters em CSV
resumo_clusters.to_csv('resumo_clusters_rfm.csv', index=False)
print("--- Arquivo resumo_clusters_rfm.csv salvo com sucesso! ---")

# 3. Leitura de confirmação (exibindo o resultado das personas no console)
with open('personas.json', 'r', encoding='utf-8') as f:
    personas_salvas = json.load(f)

print("\n--- Validação do JSON gerado: ---")
print(json.dumps(personas_salvas, indent=4, ensure_ascii=False))

--- Arquivo personas.json gerado com sucesso no disco! ---
--- Arquivo resumo_clusters_rfm.csv salvo com sucesso! ---

--- Validação do JSON gerado: ---
{
    "Clientes VIP": {
        "persona_nome": "Magnata Digital",
        "descricao_persona": "Cliente de alto poder aquisitivo, compra com frequência elevada (≈14 vezes/ano) e mantém relacionamento recente (última compra há ~12 dias). Valoriza exclusividade, atendimento premium e experiências personalizadas, demonstrando forte lealdade à marca.",
        "estrategia_retencao": "Implementar um programa de fidelidade VIP com benefícios exclusivos (acesso antecipado a lançamentos, descontos progressivos, concierge de compras) e campanhas de marketing 1‑to‑1 baseadas em histórico de compra e preferências individuais.",
        "copy_notificacao_push": "Olá {first_name}, desbloqueie agora seu acesso antecipado ao lançamento premium + 20% de desconto exclusivo para nossos clientes VIP!"
    },
    "Clientes Inativos": {
        "persona_n